In [1]:
import os
import sys
import numpy as np
import time
import scipy
import matplotlib.pyplot as plt
import time


from scipy.stats import binned_statistic
from scipy.interpolate import interp1d
from scipy.sparse.linalg import LinearOperator, splu
from scipy.sparse.linalg._isolve.utils import make_system
from scipy.sparse import csr_matrix

current_dir =  os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, os.pardir))
sys.path.append(parent_dir)

#frank2d
from frank2d import Frank2D
from constants import rad_to_arcsec, deg_to_rad
from plot import Plot
from fitting import IterativeSolverMethod
from preprocess_vis import Gridding
from geometry import Geometry

In [2]:
def frank2d_gridding(disk_name, filename,filename_output, disk_params, N):
    print("Gridding: " + disk_name + " with " + str(N)+ 'x'+str(N)+ " points" )

    inc = disk_params['inc']
    pa = disk_params['pa']
    dra = disk_params['dra']
    ddec = disk_params['ddec']
    Rout = disk_params['rout']
    
    # UVtable
    data_file = filename
    
     # load data
    data = np.load(data_file)
    u, v, Re, Imag, Weights = data['u'], data['v'], data['Re'], data['Im'], data['w']
    Vis = Re + Imag*1j
    
    geom = Geometry(inc, pa, dra, ddec)
    frank2d = Frank2D(N, Rout, geom)
    
    start_time = time.time()
    
    frank2d.preprocess_vis(u, v, Vis, Weights, hermitian= True)
    u_gridded, v_gridded, vis_gridded, weights_gridded = frank2d._gridded_data['u'], frank2d._gridded_data['v'], frank2d._gridded_data['vis'], frank2d._gridded_data['weights']

    filename_output
    with open(filename_output, 'w') as f:
        f.write("# u[lambda]    v[lambda]    Vis[Jy]    weight\n")
        
        for i in range(0, len(u_gridded)):
            f.write(f"{u_gridded[i]:.6e}    {v_gridded[i]:.6e}  {vis_gridded.real[i]:.6e}   {vis_gridded.imag[i]:.6e}   {weights_gridded[i]:.6e}\n")
            

In [3]:
dir = "../../../data/uvtables/non_axisymmetric/"

In [4]:
disk_name = "HD143006"
disk_params =  {'inc':18.6, 'pa':169 , 'dra':-5.9e-3, 'ddec':21.7e-3, 'rout':0.518*2}
# HD163296: {'inc':46.7, 'pa':133.33, 'dra':-2.8e-3, 'ddec':7.7e-3, 'rout':1.7*2}
# Elias24: {'inc': 29, 'pa': 45.7, 'dra':110.8e-3, 'ddec':-386.8e-3, 'rout': 1.03*2}
# AS209: {'inc': 34.97, 'pa': 85.76, 'dra':1.9e-3, 'ddec':-2.5e-3, 'rout': 1.2*2}
# Elias27: {'inc': 56.2, 'pa': 118.8, 'dra':-5e-3, 'ddec':-8e-3, 'rout': 1.88*2}
# IMLup: {'inc': 47.5, 'pa': 144.5, 'dra':-1.5e-3, 'ddec':1e-3, 'rout': 1.71*2}
N = 200
dir += "uvtable_"

In [5]:
frank2d_gridding(disk_name, dir + disk_name + "_continuum.npz", dir+disk_name+ "_continuum_gridded_N"+str(N)+".txt", disk_params, N )

Gridding: HD143006 with 200x200 points


/Users/mariajmelladot/Desktop/Frank2D/6_Frank2D_Oficial/frank2d/preprocess_vis.py:46: RuntimeWarning: invalid value encountered in divide
  vis_gridded_matrix =  vis_weights_sum_bin/weights_gridded_matrix


Enforcing Hermitian symmetry...
  --> time = 0.24  min |  14.67 seconds
Setting gridded data...
